In [1]:
import os
from pathlib import Path


In [2]:
%pwd

'e:\\Text-Summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Text-Summarizer'

In [ ]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    learning_rate: float

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = Path("E:/Text-Summarizer/config/config.yaml"),
        params_filepath: Path = Path("E:/Text-Summarizer/params.yaml"),
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:    
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        # ✅ EVERYTHING MUST BE INSIDE FUNCTION
        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_ckpt=config.model_ckpt,

            num_train_epochs=int(params.num_train_epochs),
            warmup_steps=int(params.warmup_steps),
            per_device_train_batch_size=int(params.per_device_train_batch_size),
            weight_decay=float(params.weight_decay),
            logging_steps=int(params.logging_steps),
            evaluation_strategy=IntervalStrategy.STEPS,
            eval_steps=int(params.eval_steps),
            save_steps=int(params.save_steps),
            gradient_accumulation_steps=int(params.gradient_accumulation_steps)
        )

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset, load_from_disk
import torch
from transformers import IntervalStrategy


e:\Text-Summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):    
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer=AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        dataset_samsum_pt=load_from_disk(self.config.data_path)
        trainer_args=TrainingArguments(
            output_dir=str(self.config.root_dir),
            num_train_epochs=int(self.config.num_train_epochs),
            warmup_steps=int(self.config.warmup_steps),
            per_device_train_batch_size=int(self.config.per_device_train_batch_size),
            weight_decay=float(self.config.weight_decay),
            logging_steps=int(self.config.logging_steps),

            evaluation_strategy=self.config.evaluation_strategy.strip().lower(),
            eval_steps=int(self.config.eval_steps),
            save_steps=int(self.config.save_steps),

            gradient_accumulation_steps=int(self.config.gradient_accumulation_steps),
        )

        trainer=Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )
        trainer.train()
        
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))
           

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(model_trainer_config)
    model_trainer.train()
except Exception as e:    
    raise e


[2026-03-26 15:29:56,683: INFO: common]: yaml file: E:\Text-Summarizer\config\config.yaml loaded successfully
[2026-03-26 15:29:56,688: INFO: common]: yaml file: E:\Text-Summarizer\params.yaml loaded successfully
[2026-03-26 15:29:56,690: INFO: common]: created directory at: artifacts
[2026-03-26 15:29:56,692: INFO: common]: created directory at: artifacts/model_trainer


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
e:\Text-Summarizer\textS\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/920 [00:00<?, ?it/s]e:\Text-Summarizer\textS\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import transformers
print(transformers.__version__)

4.41.2


In [ ]:
import sys
print(sys.executable)


e:\Text-Summarizer\textS\Scripts\python.exe


In [ ]:
print(TrainingArguments)

<class 'transformers.training_args.TrainingArguments'>


In [ ]:
import inspect
print(inspect.getfile(TrainingArguments))

e:\Text-Summarizer\textS\Lib\site-packages\transformers\training_args.py


In [ ]:
import transformers
print(transformers.__version__)

5.3.0


In [ ]:
config = ConfigurationManager()
trainer_config = config.get_model_trainer_config()

print(type(trainer_config.eval_steps))
print(type(trainer_config.save_steps))

[2026-03-25 22:10:48,003: INFO: common]: yaml file: E:\Text-Summarizer\config\config.yaml loaded successfully
[2026-03-25 22:10:48,006: INFO: common]: yaml file: E:\Text-Summarizer\params.yaml loaded successfully
[2026-03-25 22:10:48,006: INFO: common]: created directory at: artifacts
[2026-03-25 22:10:48,015: INFO: common]: created directory at: artifacts/model_trainer
<class 'int'>
<class 'int'>


In [ ]:
model_trainer.train()

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: '>' not supported between instances of 'str' and 'int'